# Day 7: Sales KPIs in PySpark
## Business Question
The Head of E-commerce asks: "Give me a monthly business review. How are we doing, where is the problem, and what should we do?"

### 1. Load and Profile
- **Dataset:** Synthetic e-commerce sales data.
- **Date Range:** 2024-01-01 - 30-10-2026
- **Total Rows:** 23,538

In [0]:
%python
from pyspark.sql import functions as F

# Load the raw data
raw_sales = spark.table("workspace.default.ecommerce_sales_flat")
print(f"Total raw rows: {raw_sales.count()}")
print(f"Total distinct orders: {raw_sales.select('order_id').distinct().count()}")

# 3. Find the date range
date_stats = sales_raw.agg(
    F.min("order_date").alias("min_date"),
    F.max("order_date").alias("max_date")
).collect()[0]

print(f"Date Range: {date_stats['min_date']} to {date_stats['max_date']}")




Total raw rows: 23538
Total distinct orders: 14000
Date Range: 2024-01-01 to 2026-10-31


### 2. Data-Quality Log and Cleaning
Here are the four business rules I applied to build `sales_clean`:
1. **Duplicates:** Removed 57 exact duplicate rows to prevent revenue inflation.
2. **Payment Spelling:** Standardized to Title Case (e.g., "upi" -> "UPI").
3. **Unmatched Products:** Kept the 8 P999 lines to maintain honest revenue, but labeled the category as "Unknown".
4. **Bad Dates:** Kept the rows, but created an `is_bad_date` flag to exclude them from delivery-time calculations.

In [0]:
%python
# 1. Drop duplicates
sales_clean = raw_sales.dropDuplicates()

# 2. Fix the category
sales_clean = sales_clean.withColumn(
    "category",
    F.when(F.col("category").isNull(), "Unknown").otherwise(F.col("category"))
)

# 3. Standardize payment methods
sales_clean = sales_clean.withColumn(
    "payment_method",
    F.initcap(F.col("payment_method"))
)

# 4. Flag bad dates
sales_clean = sales_clean.withColumn(
    "is_bad_date",
    F.when(F.col("delivery_date") < F.col("order_date"), True).otherwise(False)
)

# 5. Save the table
sales_clean.write.mode("overwrite").saveAsTable("workspace.default.sales_clean")

print(f"Clean rows saved: {sales_clean.count()}")

Clean rows saved: 23508


### 3. KPI Definitions
- **Net Revenue:** Sum of `line_amount` where `order_status == 'Delivered'`.
- **GMV:** Sum of `line_amount` across ALL statuses.
- **Average Order Value (AOV):** Net Revenue ÷ Distinct `order_id`.
- **Gross Margin %:** Total Margin ÷ Total Revenue (excluding null unit costs).
- **Cancellation Rate:** Cancelled Orders ÷ Total Orders.
- **Return Rate:** Returned Orders ÷ Delivered Orders.
- **Average Discount:** Total Discount Amount ÷ Total Gross Amount.
- **Delivery Days:** Average days between order and delivery (excluding `is_bad_date = True`).
- **Repeat Customer Rate:** Customers with >1 order ÷ Total Customers.

In [0]:
%python
# First, let's create the margin and discount amounts since they are missing from the raw file
sales_clean = sales_clean.withColumn(
    "Margin_amount", 
    F.col("line_amount") - (F.col("unit_cost") * F.col("quantity"))
).withColumn(
    "discount_amount", 
    (F.col("unit_price") * F.col("quantity")) - F.col("line_amount")
).withColumn(
    "gross_amount", 
    F.col("unit_price") * F.col("quantity")
)

# Now run the KPI calculations
kpi_stats = sales_clean.agg(
    F.sum("line_amount").alias("gmv"),
    F.sum(F.when(F.col("order_status") == "Delivered", F.col("line_amount"))).alias("net_revenue"),
    F.countDistinct("order_id").alias("total_orders"),
    F.countDistinct(F.when(F.col("order_status") == "Delivered", F.col("order_id"))).alias("delivered_orders"),
    F.countDistinct(F.when(F.col("order_status") == "Cancelled", F.col("order_id"))).alias("cancelled_orders"),
    F.countDistinct(F.when(F.col("order_status") == "Returned", F.col("order_id"))).alias("returned_orders"),
    
    # Margin and discount calculations
    F.sum(F.when(F.col("unit_cost").isNotNull(), F.col("Margin_amount"))).alias("total_margin"),
    F.sum(F.when(F.col("unit_cost").isNotNull(), F.col("line_amount"))).alias("valid_revenue_for_margin"),
    F.sum("discount_amount").alias("total_discount"),
    F.sum("gross_amount").alias("total_gross")
).collect()[0]

# Do the math
net_rev = kpi_stats["net_revenue"]
aov = net_rev / kpi_stats["total_orders"]
margin_pct = (kpi_stats["total_margin"] / kpi_stats["valid_revenue_for_margin"]) * 100
cancel_rate = (kpi_stats["cancelled_orders"] / kpi_stats["total_orders"]) * 100
return_rate = (kpi_stats["returned_orders"] / kpi_stats["delivered_orders"]) * 100
avg_discount = (kpi_stats["total_discount"] / kpi_stats["total_gross"]) * 100

# Print
print(f"--- HEADLINE NUMBERS ---")
print(f"GMV:                 ₹{kpi_stats['gmv']:,.2f}")
print(f"Net Revenue:         ₹{net_rev:,.2f}")
print(f"Average Order Value: ₹{aov:,.2f}")
print(f"Gross Margin %:      {margin_pct:.2f}%")
print(f"Cancellation Rate:   {cancel_rate:.2f}%")
print(f"Return Rate:         {return_rate:.2f}%")
print(f"Average Discount:    {avg_discount:.2f}%")

--- HEADLINE NUMBERS ---
GMV:                 ₹166,638,705.40
Net Revenue:         ₹143,116,773.70
Average Order Value: ₹10,222.63
Gross Margin %:      26.60%
Cancellation Rate:   7.85%
Return Rate:         7.26%
Average Discount:    9.69%


### 4. KPI Results

**Headline Numbers (All-Time):**
- **GMV:** ₹16.66 Crore
- **Net Revenue:** ₹14.31 Crore
- **Average Order Value (AOV):** ₹10,222
- **Gross Margin:** 26.60%
- **Cancellation Rate:** 7.85%
- **Return Rate:** 7.26%
- **Average Discount:** 9.69%

In [0]:
%python
from pyspark.sql.window import Window

# 1. Monthly Trend
monthly_trend = sales_clean.filter(F.col("order_status") == "Delivered") \
    .groupBy(F.year("order_date").alias("year"), F.month("order_date").alias("month")) \
    .agg(F.sum("line_amount").alias("revenue")) \
    .orderBy("year", "month")

# Add MoM Growth
w = Window.orderBy("year", "month")
monthly_trend = monthly_trend.withColumn(
    "mom_growth_pct",
    F.round(((F.col("revenue") - F.lag("revenue", 1).over(w)) / F.lag("revenue", 1).over(w)) * 100, 2)
)
print("--- MONTHLY TREND (Top 5 rows) ---")
monthly_trend.show(5)

--- MONTHLY TREND (Top 5 rows) ---


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+----+-----+------------------+--------------+
|year|month|           revenue|mom_growth_pct|
+----+-----+------------------+--------------+
|2024|    1| 3190161.900000001|          NULL|
|2024|    2|2180817.7500000005|        -31.64|
|2024|    3|        2414663.55|         10.72|
|2024|    4| 2525880.100000001|          4.61|
|2024|    5|         3011151.4|         19.21|
+----+-----+------------------+--------------+
only showing top 5 rows


In [0]:
# 2. Category View
category_view = sales_clean.filter(F.col("order_status") == "Delivered") \
    .groupBy("category") \
    .agg(
        F.sum("line_amount").alias("revenue"),
        F.sum("Margin_amount").alias("margin")
    ) \
    .withColumn("share_pct", F.round((F.col("revenue") / 143116773.70) * 100, 2)) \
    .withColumn("margin_pct", F.round((F.col("margin") / F.col("revenue")) * 100, 2)) \
    .orderBy(F.col("revenue").desc())

print("--- CATEGORY VIEW ---")
category_view.show()

--- CATEGORY VIEW ---
+--------------+--------------------+--------------------+---------+----------+
|      category|             revenue|              margin|share_pct|margin_pct|
+--------------+--------------------+--------------------+---------+----------+
|   Electronics| 7.630502250000001E7|1.1958243500000002E7|    53.32|     15.67|
|Home & Kitchen|3.2966404500000022E7|1.0240792499999978E7|    23.03|     31.06|
|       Fashion|1.9393173200000003E7|   9662890.200000007|    13.55|     49.83|
|        Sports|   8164949.199999998|  3099057.1999999993|     5.71|     37.96|
|        Beauty|           4893228.1|  2484361.1000000015|     3.42|     50.77|
|         Books|  1344091.4499999974|  341617.45000000077|     0.94|     25.42|
|       Unknown|            49904.75|                NULL|     0.03|      NULL|
+--------------+--------------------+--------------------+---------+----------+



In [0]:
# 3. Region View
region_view = sales_clean.filter(F.col("order_status") == "Delivered") \
    .groupBy("region") \
    .agg(F.sum("line_amount").alias("revenue")) \
    .orderBy(F.col("revenue").desc())

print("--- REGION VIEW ---")
region_view.show()

--- REGION VIEW ---
+------+--------------------+
|region|             revenue|
+------+--------------------+
| South|4.1951753300000064E7|
|  West| 4.187903480000007E7|
| North| 3.650296519999998E7|
|  East| 2.278302040000003E7|
+------+--------------------+



In [0]:
%python
# 1. Isolate North + Fashion
north_fashion = sales_clean.filter(
    (F.col("region") == "North") & 
    (F.col("category") == "Fashion")
)

# 2. Calculate the return rate and rupee value
nf_stats = north_fashion.agg(
    F.countDistinct(F.when(F.col("order_status") == "Delivered", F.col("order_id"))).alias("delivered_orders"),
    F.countDistinct(F.when(F.col("order_status") == "Returned", F.col("order_id"))).alias("returned_orders"),
    F.sum(F.when(F.col("order_status") == "Returned", F.col("line_amount"))).alias("returned_value")
).collect()[0]

nf_return_rate = (nf_stats["returned_orders"] / nf_stats["delivered_orders"]) * 100

print(f"--- DEEP DIVE: North Fashion ---")
print(f"Delivered Orders: {nf_stats['delivered_orders']}")
print(f"Returned Orders:  {nf_stats['returned_orders']}")
print(f"Return Rate:      {nf_return_rate:.2f}%")
print(f"Rupee Value Lost: ₹{nf_stats['returned_value']:,.2f}")

--- DEEP DIVE: North Fashion ---
Delivered Orders: 1125
Returned Orders:  253
Return Rate:      22.49%
Rupee Value Lost: ₹1,071,726.85


### 5. Deep Dive: North Fashion Returns
Fashion has a return rate of **[22.49%]** (vs 7.26%). The total cost of returns for this category is **[₹1,071,726.85].**

As there is no column for reason for return we can’t know what the cause of returns is. We recommend creating a mandatory drop down for returning customers to fill in why they are returning their item in order to understand if it is size / quality related.

### 6. Recommendations and Limitations

**Recommendations:**
1. **Push Beauty and Fashion Bundles:** Promote bundles in Beauty/Fashion categories. As Electronics only makes up a 15.6% margin, we can increase our profitability by promoting higher margin (50.7% and 49.8%) Beauty and Fashion products as bundles at checkout. This will raise our overall Gross Margin to >30%.*
2. **Investigate the East Region:** East Region. Compared to the South region, we are losing almost 2crore in the East. It would be worthwhile for the Marketing department to run a targeted ad campaign in the East Region. Alternatively, it would be worth checking with the supply chain team if we are understocking high value electronics in eastern warehouses.
3. **Launch a Retention Campaign in February:** Retention Strategy in February. Revenue drops significantly in February (31.6% drop). A targeted promotion for Valentine’s Day/New Year’s Day early in February could help soften this drop.

**Limitations:**
- The data is fabricated and therefore may not accurately represent real customer data.
- We have no information regarding reasons for returns meaning reasons for returns can only be assumed.
- 30 rows were removed as they were exact duplicates according to our criteria. This may cause revenue to be slightly underestimated.